# Aether Stage 2 — Phase 0 engineering harness

Mandatory Phase 0 from `docs/stage2_spec.md`: Qwen3-0.6B, direct speech-state to transcript generation, frozen Stage 1 encoder and frozen Qwen, trainable Connector only. This is a software-stack validation, not a quality result.

Artifacts are written to a new timestamped directory under `MyDrive/aether-v3/stage2/`. The Hugging Face token is read from the Colab secret `HF_TOKEN`.


In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", force=True)

import os, subprocess, sys, logging, time
from pathlib import Path
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
assert os.environ["HF_TOKEN"], "Add HF_TOKEN in Colab Secrets and enable notebook access"

REPO_URL = "https://github.com/karl4th/aether-v3.git"
REPO_DIR = "/content/aether-v3"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", "stage2"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", "stage2", "--single-branch", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "stage2"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/stage2"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR, "huggingface_hub", "pytest", "ruff", "mypy"], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
# A rerun may follow a git update in the same kernel. Drop stale project modules.
for module_name in list(sys.modules):
    if module_name == "aether_v3" or module_name.startswith("aether_v3."):
        del sys.modules[module_name]
print(subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

## 1. Repository tests

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

## 2. Load the private Stage 1 encoder

In [ ]:
CONFIG_PATH = "configs/stage2_smoke.yaml"
import inspect
import aether_v3.data.stage2_cache as stage2_cache_module
source = inspect.getsource(stage2_cache_module.build_slue_sqa5_shards)
assert "build_limmim_stage2_shards" in inspect.getsource(stage2_cache_module), "Old Stage 2 cache module loaded"
print("Stage 2 source preflight passed")
import torch
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer

from aether_v3.config import load_config
from aether_v3.models.aether_speech import AetherSpeechEncoder
from aether_v3.training.stage2_utils import load_stage1_encoder

cfg = load_config(CONFIG_PATH)
assert torch.cuda.is_available(), "Stage 2 requires a GPU runtime"
if cfg.llm.dtype == "bfloat16" and not torch.cuda.is_bf16_supported():
    print("GPU has no native BF16 support; using float16")
    cfg.llm.dtype = "float16"
    cfg.stage2_train.amp_dtype = "float16"
print("GPU:", torch.cuda.get_device_name(0), "LLM dtype:", cfg.llm.dtype)
stage1_path = hf_hub_download(
    repo_id=cfg.stage2_train.stage1_repo_id,
    filename=cfg.stage2_train.stage1_filename,
    revision=cfg.stage2_train.stage1_revision,
    token=os.environ["HF_TOKEN"],
)
encoder = AetherSpeechEncoder(cfg.aether_speech)
checkpoint = load_stage1_encoder(stage1_path, encoder)
encoder.eval()
print("Stage 1 encoder loaded; checkpoint step:", checkpoint.get("step", "unknown"))

## 3. Build the Phase 0 transcription cache

Loads semantic Mimi q0 codes and byte transcripts from `karl4th/limmim`, then runs the frozen Stage 1 encoder. No audio download or Mimi pass is repeated. Progress is logged after every encoder batch.


In [ ]:
from datasets import load_dataset
from aether_v3.data.stage2_cache import build_limmim_stage2_shards
from aether_v3.training.stage2_utils import create_run_dir

run_dir = create_run_dir(cfg.stage2_train.drive_root)
cache_root = run_dir / "cache"
print("PHASE 0 RUN:", run_dir, flush=True)
print("Loading karl4th/limmim streams...", flush=True)
train_rows = load_dataset("karl4th/limmim", split="train", streaming=True, token=os.environ["HF_TOKEN"])
val_rows = load_dataset("karl4th/limmim", split="validation", streaming=True, token=os.environ["HF_TOKEN"])
tokenizer = AutoTokenizer.from_pretrained(cfg.llm.model_id, revision=cfg.llm.revision)
print("Building 64-example train cache...", flush=True)
build_limmim_stage2_shards(train_rows, encoder, tokenizer, cache_root/"train", "train", max_examples=64)
print("Building 16-example validation cache...", flush=True)
build_limmim_stage2_shards(val_rows, encoder, tokenizer, cache_root/"validation", "validation", max_examples=16)
print("Phase 0 caches ready", flush=True)


## 4. Run Phase 0

Runs mandatory `eval@step0`, ten optimizer steps, evaluations at steps 5 and 10, transcript generation, WER/CER, and all checkpoint paths. Console output reports every step, GPU memory, evaluations, generated examples, and checkpoint saves.


In [ ]:
from aether_v3.models.aether_speech_llm import AetherSpeechLLM
from aether_v3.training.train_stage2 import run_stage2_training

print("Loading Qwen3-0.6B...", flush=True)
model = AetherSpeechLLM(cfg.aether_speech, cfg.connector, cfg.llm, speech_frozen=True)
load_stage1_encoder(stage1_path, model.encoder)
print("Starting Phase 0 training loop...", flush=True)
run_stage2_training(cfg, run_dir, cache_root/"train", cache_root/"validation", model=model, tokenizer=tokenizer)
required = ["last.pt", "best_val_loss.pt", "best_wer.pt", "best_cer.pt"]
for name in required:
    assert (run_dir/name).exists(), f"Missing {name}"
print("PHASE 0 PASSED:", run_dir, flush=True)


## Pass criteria

Phase 0 passes when the complete direct-speech transcription path runs, Connector gradients update, generation produces decoded hypotheses, WER/CER are measured at step 0/5/10, and `last.pt`, periodic, `best_val_loss.pt`, `best_wer.pt`, and `best_cer.pt` exist on Drive. Scores are not a quality claim at this phase.
